# Exercise 0 - PydanticAI and OpenRouter
In the data folder, there are a few job ads taken from arbetsförmedlingen.se.

a) Read all the jobs ads into python

In [1]:
with open("data/ads1.txt", "r") as file:
    ads1 = file.read()

with open("data/ads2.txt", "r") as file:
    ads2 = file.read()

with open("data/ads3.txt", "r") as file:
    ads3 = file.read()

b) Create a function that uses PydanticAI agent to summarize a job ad. This function should take in an ad as its parameter and return a summary.

In [2]:
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

async def summarize_job_ad(ad: str) -> str:
    agent = Agent(
        "openrouter:arcee-ai/trinity-large-preview:free",
        system_prompt="""You are a job ad summarizer. You will be given a job ad and you will need to summarize it in a few sentences.
        
    Areas to include in the output:
    - summary
    - job title
    - salary SEK per month
    
    make the output in markdown format""",
        retries=1,
    )
    result = await agent.run(ad)
    return result.output

await summarize_job_ad(ads1)



"The Data Platform team at Instabee is seeking a Data Engineer with an interest in Analytics Engineering to help shape the company's data platform. This role involves building and migrating data integrations, setting up data platform infrastructure, and introducing data product principles. The ideal candidate will have experience in both Data Engineering and Analytics Engineering, with a university degree in a technical or analytical field. The position offers a modern tech stack, flexible hours, and a range of unique benefits, including a sky-high office with a 360-degree view of Stockholm and a puppy-friendly workplace. The salary is not specified in the job ad."

c) Now create and export markdown files for each job ad and its corresponding summary.

In [3]:
with open("data/ads1.md", "w") as file:
    file.write(ads1)

with open("data/ads2.md", "w") as file:
    file.write(ads2)

with open("data/ads3.md", "w") as file:
    file.write(ads3)

d) Try to take other job ads from arbetsförmedlingen.se and see how well your function performs.

In [4]:
with open("data/arbetsfomedlingen.txt", "r") as file:
    arbete = file.read()

await summarize_job_ad(arbete)


'The job ad is for a Machine Learning / MLOps Engineer position at Gears Of Leo AB in Stockholm. The role involves designing, implementing, and maintaining scalable MLOps pipelines, building cloud infrastructure for ML models, and collaborating with data scientists to ensure production readiness. The ideal candidate should have mid-senior level experience in Machine Learning or MLOps, strong proficiency in Python and cloud platforms, and a solid understanding of CI/CD pipelines. The salary is not specified in the ad. Benefits include a hybrid work policy, 30 annual vacation days, wellness contribution, and more.'

e) Make the output structured using a pydantic model together with PydanticAI agent. The output from the agent should have the following fields:

- job_title
- description
- summary
- responsibilities
- words_in_article

In [5]:
from pydantic import BaseModel, Field

class JobAdSummary(BaseModel):
    job_title: str = Field(description="The title of the job ad")
    description: str = Field(description="A description of the job ad")
    summary: str = Field(description="A summary of the job ad")
    responsibilities: str = Field(description="The responsibilities of the job ad")
    words_in_article: int = Field(description="The number of words in the job ad")

job_ad_agent = Agent(
    "openrouter:nvidia/nemotron-3-super-120b-a12b:free",
    system_prompt="You are a job ad summarizer. You will be given a job ad and you will need to summarize it in a few sentences.",
    retries=1,
)

result = await job_ad_agent.run(arbete, output_type=JobAdSummary)


Traceback (most recent call last):
  File "/Users/lucaslindh/MLOps/github/llmops_lucas_lindh_mlo25/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3745, in run_code
    await eval(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/pt/91l37zls1kn7rrbgk9l8k5v00000gn/T/ipykernel_93408/1311448519.py", line 16, in <module>
    result = await job_ad_agent.run(arbete, output_type=JobAdSummary)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/lucaslindh/MLOps/github/llmops_lucas_lindh_mlo25/.venv/lib/python3.13/site-packages/pydantic_ai/agent/abstract.py", line 331, in run
    node = await agent_run.next(node)  # pyright: ignore[reportArgumentType]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/lucaslindh/MLOps/github/llmops_lucas_lindh_mlo25/.venv/lib/python3.13/site-packages/pydantic_ai/run.py", line 356, in next
    return await self._run_node_with_hooks(node, self._advance_graph)
           ^^^^^^^^^^^^^^^^

In [ ]:
result.output.model_dump()

{'job_title': 'Machine Learning / MLOps Engineer',
 'description': 'Gears Of Leo AB is seeking a Machine Learning / MLOps Engineer to join their newly formed MLOps and ML Engineering team in Stockholm. The role focuses on building scalable ML infrastructure, implementing MLOps pipelines, managing cloud infrastructure on GCP, establishing model monitoring, and collaborating with data scientists to ensure production readiness of ML models.',
 'summary': 'Machine Learning / MLOps Engineer position at Gears Of Leo AB in Stockholm, responsible for designing and maintaining MLOps pipelines, building GCP infrastructure, implementing model monitoring, and collaborating with data scientists to productionize ML models.',
 'responsibilities': 'Designing, implementing, and maintaining scalable MLOps pipelines for model training, testing, deployment, and monitoring; building and managing cloud infrastructure on GCP for real-time and batch ML models; establishing robust model monitoring (drift, perf

In [ ]:
ad = result.output

import pandas as pd

df = pd.DataFrame([ad.model_dump()])
df



,job_title,description,summary,responsibilities,words_in_article
0,Machine Learning / MLOps Engineer,Gears Of Leo AB is seeking a Machine Learning ...,Machine Learning / MLOps Engineer position at ...,"Designing, implementing, and maintaining scala...",480
